In [9]:
import requests
from bs4 import BeautifulSoup, Comment
from urllib.parse import urljoin
import time

In [10]:
# トップページとドメイン
start_url = "https://www.musashino-u.ac.jp/"
domain = "musashino-u.ac.jp"

# 訪問管理と結果保存
to_visit = [start_url]   # これから訪問するページ
visited = set()          # 訪問済みページ
result = {}              # {URL: <title>文字列}


In [11]:
while to_visit:
    url = to_visit.pop(0)
    if url in visited:
        continue

    print(f"アクセス中: {url}")
    try:
        response = requests.get(url, timeout=10)
        response.encoding = response.apparent_encoding
    except Exception as e:
        print(f"アクセス失敗: {url} ({e})")
        continue

    visited.add(url)
    soup = BeautifulSoup(response.text, "html.parser")

    # コメントアウト削除（コメント内リンク除外）
    for c in soup.find_all(text=lambda t: isinstance(t, Comment)):
        c.extract()

    # titleタグ取得（存在しない場合スキップ）
    title_tag = soup.find("title")
    if not title_tag or not title_tag.get_text().strip():
        print(f"スキップ: {url}（titleなし）")
        continue

    title_text = title_tag.get_text().strip()
    result[url] = title_text

    # aタグ探索（相対パス含む）
    for a_tag in soup.find_all("a", href=True):
        link = a_tag["href"].strip()

        # 無効・不要なリンクを除外
        if link.startswith(("javascript:", "mailto:", "tel:", "#")):
            continue
        if not link:
            continue

        # 相対パスを絶対URLへ変換
        full_link = urljoin(url, link)

        # 外部サイト・不要ファイル除外
        if domain not in full_link:
            continue
        if any(ext in full_link for ext in [".pdf", ".jpg", ".png", ".mp4", ".zip"]):
            continue

        if full_link not in visited and full_link not in to_visit:
            to_visit.append(full_link)

    # サーバーへの負荷軽減
    time.sleep(1)

    # 上限設定（安全のため）
    if len(result) >= 200:
        print("200ページに到達したため終了します。")
        break


アクセス中: https://www.musashino-u.ac.jp/


/var/folders/ts/sk8mfy_x6cl0pg0hrmp0mfdr0000gn/T/ipykernel_99771/2099816543.py:18: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  for c in soup.find_all(text=lambda t: isinstance(t, Comment)):


アクセス中: https://ef.musashino-u.ac.jp/donation/
アクセス中: https://www.musashino-u.ac.jp/access.html
アクセス中: https://www.musashino-u.ac.jp/admission/request.html
アクセス中: https://www.musashino-u.ac.jp/contact.html
アクセス中: https://www.musashino-u.ac.jp/prospective-students.html
アクセス中: https://www.musashino-u.ac.jp/students.html
アクセス中: https://www.musashino-u.ac.jp/alumni.html
アクセス中: https://www.musashino-u.ac.jp/parents.html
アクセス中: https://www.musashino-u.ac.jp/business.html
アクセス中: https://www.musashino-u.ac.jp/guide/
アクセス中: https://www.musashino-u.ac.jp/guide/profile/
アクセス中: https://www.musashino-u.ac.jp/guide/activities/
アクセス中: https://www.musashino-u.ac.jp/guide/campus/
アクセス中: https://www.musashino-u.ac.jp/guide/facility/
アクセス中: https://www.musashino-u.ac.jp/guide/information/
アクセス中: https://www.musashino-u.ac.jp/guide/profile/media/
アクセス中: https://www.musashino-u.ac.jp/admission/
アクセス中: https://www.musashino-u.ac.jp/admission/faculty/
アクセス中: https://www.musashino-u.ac.jp/admission/internation

In [12]:
print(result)

{'https://www.musashino-u.ac.jp/': '武蔵野大学', 'https://ef.musashino-u.ac.jp/donation/': 'ご寄付のお願い | 学校法人武蔵野大学', 'https://www.musashino-u.ac.jp/access.html': '交通アクセス | 武蔵野大学', 'https://www.musashino-u.ac.jp/admission/request.html': '資料請求 | 入試情報 | 武蔵野大学', 'https://www.musashino-u.ac.jp/contact.html': 'お問い合わせ | 武蔵野大学', 'https://www.musashino-u.ac.jp/prospective-students.html': '武蔵野大学で学びたい方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/students.html': '在学生の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/alumni.html': '卒業生の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/parents.html': '保護者の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/business.html': '企業・研究者の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/': '大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/profile/': '大学紹介 | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/activities/': '大学の取り組み | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/campus/': 'キャンパス | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/facility/': '附置機関・センター・附属施設 | 大学案内 